In [25]:
from neuroconv.datainterfaces import IntanDigitalInterface

path = r"E:\Backup\Akseli\AI_data\rawdata\sub-01_id-Ivy\ses-000_date-20250309_01\ephys\Ivy_250309_111721\info.rhd"
path2 = r"E:\Backup\Akseli\AK_data\rawdata\sub-03_id-Freddy\ses-000_date-20250528_01\ephys\Freddy_250528_112348\info.rhd"
digital_interface = IntanDigitalInterface(file_path=path2)
digital_interface.get_metadata()

DeepDict(
{'NWBFile': {'session_description': '',
  'identifier': 'bfbd83a1-07dd-4a9c-92a3-f397151b038d',
  'source_script': 'NeuroConv provenance record\nneuroconv_version: 0.10.2.dev0\nexecution_environment: notebook\nneuroconv_provenance_format: 1',
  'source_script_file_name': 'neuroconv'},
 'Events': {'intan_digital': {'event_types': {'DIGITAL-IN-01': {'event_name': 'DIGITAL-IN-01'}}}}}
)

In [42]:

digital_interface = IntanDigitalInterface(
    file_path=path,
    detection_configuration={
        "DIGITAL-IN-01": [
            {
                "signal_conditioning": {"binarize": "midpoint"},
                "detection": "rising",
                "event_name": "camera_frame",
            }
        ]
    },
)

frame_pulse_times = digital_interface.get_event_times("camera_frame")

In [45]:
from neuroconv.datainterfaces import ExternalVideoInterface
from pathlib import Path

video_extensions = {".avi", ".mp4", ".mov", ".mkv"}
folder = Path(r"C:\Users\aksel\Documents\VidData\20250309_01_Ivy")
folder2 = Path(r"C:\Users\aksel\Documents\VidData\20250528_01_Freddy")

file_paths = sorted(
    p for p in folder.glob("*cam-1*") if p.suffix.lower() in video_extensions
)

video_interface = ExternalVideoInterface(
    file_paths=file_paths,
    video_name="CameraA",
)


In [ ]:
file_paths

WindowsPath('C:/Users/aksel/Documents/VidData/20250528_01_Freddy/2025-05-28_136_Freddy-cam-1.mp4')

In [38]:
durations = np.array(interface.get_header_frame_counts()) / np.array(interface.get_header_frame_rates())
durations

array([ 5.675,  5.34 ,  5.465,  5.775,  4.575,  5.415,  5.685,  5.705,
        4.63 ,  6.18 ,  4.81 ,  5.13 ,  5.505,  5.355,  4.635,  5.395,
        6.08 ,  4.98 ,  5.985,  5.77 ,  4.87 ,  6.215,  7.145,  5.895,
        4.865,  7.225,  7.61 ,  6.385,  5.23 ,  5.01 ,  5.475,  4.955,
        5.13 ,  6.13 ,  5.705,  5.3  ,  6.775,  6.1  , 20.035,  7.95 ,
       11.285, 14.48 , 20.03 , 10.565,  9.685, 11.855, 11.925, 16.495,
       16.97 , 12.095,  6.885,  5.96 , 20.035, 11.46 ,  6.685,  5.885,
       14.495, 20.03 , 17.945,  7.345, 20.03 , 20.035, 18.415, 20.035,
        6.67 , 20.035, 19.63 ,  5.22 ,  5.965,  6.18 ,  7.72 ,  7.025,
        7.765,  6.23 ,  5.37 , 12.445,  8.51 ,  6.97 , 12.51 ,  9.355,
       18.23 , 10.045, 10.74 , 12.39 , 14.825, 12.65 ,  6.72 ,  6.24 ,
        9.455,  8.84 , 10.525, 11.59 ,  6.89 ,  5.93 , 10.335,  6.65 ,
        8.06 ,  6.725,  8.585, 10.215, 17.06 , 18.725, 14.36 , 17.16 ,
       20.035, 20.03 , 20.035, 20.035, 15.7  ,  9.155, 20.03 ,  6.145,
      

In [ ]:
import numpy as np




nwbfile.add_trial_column(name="video_file", description="The external_file entry holding this trial's frames.")
for onset, duration, file_path in zip(trial_onsets, durations, file_paths):
    nwbfile.add_trial(start_time=onset, stop_time=onset + duration, video_file=str(file_path))

In [ ]:
import numpy as np

frame_pulse_times# The gap between trials is far larger than the frame interval, so the split is unambiguous.
frame_interval = np.median(np.diff(frame_pulse_times))
gap_indices = np.flatnonzero(np.diff(frame_pulse_times) > 10 * frame_interval) + 1
bursts = np.split(frame_pulse_times, gap_indices)

segment_keys = video_interface.alignment.keys()

assert len(bursts) == len(segment_keys), f"{len(bursts)} bursts for {len(segment_keys)} files."
for segment_key, burst in zip(segment_keys, bursts):
    video_interface.alignment[segment_key].set_times(burst)

trial_onsets = [burst[0] for burst in bursts]

In [55]:
import numpy as np

frame_pulse_times = digital_interface.get_event_times("camera_frame")
frame_interval = np.median(np.diff(frame_pulse_times))
gap_indices = np.flatnonzero(np.diff(frame_pulse_times) > 10 * frame_interval) + 1
bursts = np.split(frame_pulse_times, gap_indices)  # 144 bursts — one per triggered trial

# From your Arduino log: 0-based indices, into the full 144-trial sequence,
# of trials whose video never reached disk.
dropped_trial_indices = {43, 44, 45, 46, 47, 48}  # whatever your Arduino rig reports

kept_bursts = [burst for i, burst in enumerate(bursts) if i not in dropped_trial_indices]

segment_keys = video_interface.alignment.keys()  # 138 — only the files that exist

assert len(kept_bursts) == len(segment_keys), f"{len(kept_bursts)} bursts for {len(segment_keys)} files."
for segment_key, burst in zip(segment_keys, kept_bursts):
    video_interface.alignment[segment_key].set_times(burst)


trial_onsets = [burst[0] for burst in bursts]

In [58]:
segment_key

'2025-03-09_144_Ivy-cam-1'

In [70]:
burst = burst[:-1] 
video_interface.alignment[segment_key].set_times(burst)

In [71]:
len(burst)

2999

In [61]:
len(video_interface.alignment[segment_key]._times)

3001

In [ ]:
import ethograph as eto

138